In [1]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask
import geopandas as gpd
import os

In [2]:
# ---- PATHS ----
raw_folder = r"D:\\Important\\Semester\\Semester X\\MTP\\Raw data"
lsi_folder = r"D:\\Important\\Semester\\Semester X\\MTP\\LSI"
shapefile_path = r"D:\\Important\\Semester\\Semester X\\MTP\\DATA\\shimla shp\\shimla_boundary_utm.shp"

In [3]:
feature_files = {
    "landslide_dist": os.path.join(raw_folder, "landslide_distance.tif"),
    "slope": os.path.join(raw_folder, "slope.tif"),
    "lulc": os.path.join(raw_folder, "lulc_shimla.tif"),
    "elevation": os.path.join(raw_folder, "dem_clip.tif"),
    "road_dist": os.path.join(raw_folder, "roads_distance.tif"),
    "builtup_dist": os.path.join(raw_folder, "distance_from_builtup.tif"),
    "land_value": os.path.join(raw_folder, "maskedlandcords.tif"),
    "water_dist": os.path.join(raw_folder, "distance_from_water.tif"),
    "lineament_dist": os.path.join(raw_folder, "lineament_aligned.tif"),
    "lst": os.path.join(raw_folder, "lst_celcius.tif"),
    "railway_dist": os.path.join(raw_folder, "railway_distance.tif"),
}

In [4]:
# ---- TARGET VARIABLES ----
target_files = {
    "lsi_continuous": os.path.join(lsi_folder, "LSI_continuous.tif"),
    "lsi_classified": os.path.join(lsi_folder, "LSI_classified.tif"),
}

In [5]:
# ---- LOAD SHAPEFILE ----
shimla_boundary = gpd.read_file(shapefile_path)

In [6]:
# ---- VERIFY SHAPEFILE AREA ----
shimla_boundary_projected = shimla_boundary.to_crs("EPSG:32643")
area_sqm = shimla_boundary_projected.geometry.area.sum()
area_sqkm = area_sqm / 1_000_000

print(f"Shapefile area: {area_sqm:,.0f} sq.m")
print(f"Shapefile area: {area_sqkm:,.2f} sq.km")
print(f"Expected study area: 124 sq.km")
print(f"Ratio: {area_sqkm/124:.1f}x larger than expected")

Shapefile area: 5,143,686,252 sq.m
Shapefile area: 5,143.69 sq.km
Expected study area: 124 sq.km
Ratio: 41.5x larger than expected


In [7]:
# ---- VERIFY ----
print("Shapefile CRS:", shimla_boundary.crs)
print("Shapefile bounds:\n", shimla_boundary.total_bounds)
print("\nFeature files found:")
for name, path in feature_files.items():
    exists = os.path.exists(path)
    print(f"  {name}: {'✓' if exists else '✗ MISSING'}")

print("\nTarget files found:")
for name, path in target_files.items():
    exists = os.path.exists(path)
    print(f"  {name}: {'✓' if exists else '✗ MISSING'}")

Shapefile CRS: EPSG:32643
Shapefile bounds:
 [ 690008.69041125 3406562.32289832  815469.52705558 3512110.81741804]

Feature files found:
  landslide_dist: ✓
  slope: ✓
  lulc: ✓
  elevation: ✓
  road_dist: ✓
  builtup_dist: ✓
  land_value: ✓
  water_dist: ✓
  lineament_dist: ✓
  lst: ✓
  railway_dist: ✓

Target files found:
  lsi_continuous: ✓
  lsi_classified: ✓


In [8]:
def extract_masked_values(raster_path, shapes):
    """Read raster, mask with shapefile, return flattened float array."""
    with rasterio.open(raster_path) as src:
        # Use filled=False to get masked array, then convert ourselves
        out_image, out_transform = mask(src, shapes, crop=True, filled=False)
        band = out_image[0].astype(np.float32)
        # Convert masked positions to NaN
        band[out_image[0].mask] = np.nan
    return band, out_transform

In [9]:
# ---- GET SHAPEFILE GEOMETRY ----
shapes = shimla_boundary.geometry.values

In [10]:
# ---- EXTRACT FIRST RASTER TO GET REFERENCE SHAPE ----
print("Extracting reference raster (landslide_dist)...")
ref_band, ref_transform = extract_masked_values(feature_files["landslide_dist"], shapes)
ref_shape = ref_band.shape
print(f"  Masked raster shape: {ref_shape}")
print(f"  Total pixels in masked area: {ref_band.size}")
print(f"  Non-NaN pixels: {np.count_nonzero(~np.isnan(ref_band))}")

Extracting reference raster (landslide_dist)...
  Masked raster shape: (10553, 12545)
  Total pixels in masked area: 132387385
  Non-NaN pixels: 51436833


In [11]:
# ---- EXTRACT ALL FEATURES ----
print("\nExtracting all feature rasters...")
data = {}

# ---- CREATE PIXEL COORDINATE ARRAYS ----
nrows, ncols = ref_shape
row_indices, col_indices = np.meshgrid(np.arange(nrows), np.arange(ncols), indexing='ij')
data["pixel_row"] = row_indices.flatten().astype(np.int32)
data["pixel_col"] = col_indices.flatten().astype(np.int32)
print(f"  Added pixel coordinates: rows 0-{nrows-1}, cols 0-{ncols-1}")

for name, path in feature_files.items():
    band, _ = extract_masked_values(path, shapes)
    if band.shape != ref_shape:
        print(f"  ⚠️ {name}: shape {band.shape} does NOT match reference {ref_shape}")
    else:
        print(f"  ✓ {name}: shape {band.shape} — OK")
    data[name] = band.flatten()


Extracting all feature rasters...
  Added pixel coordinates: rows 0-10552, cols 0-12544
  ✓ landslide_dist: shape (10553, 12545) — OK
  ✓ slope: shape (10553, 12545) — OK
  ✓ lulc: shape (10553, 12545) — OK
  ✓ elevation: shape (10553, 12545) — OK
  ✓ road_dist: shape (10553, 12545) — OK
  ✓ builtup_dist: shape (10553, 12545) — OK
  ✓ land_value: shape (10553, 12545) — OK
  ✓ water_dist: shape (10553, 12545) — OK
  ✓ lineament_dist: shape (10553, 12545) — OK
  ✓ lst: shape (10553, 12545) — OK
  ✓ railway_dist: shape (10553, 12545) — OK


In [12]:
# ---- EXTRACT TARGETS ----
print("\nExtracting target rasters...")
for name, path in target_files.items():
    band, _ = extract_masked_values(path, shapes)
    if band.shape != ref_shape:
        print(f"  ⚠️ {name}: shape {band.shape} does NOT match reference {ref_shape}")
    else:
        print(f"  ✓ {name}: shape {band.shape} — OK")
    data[name] = band.flatten()

print(f"\nTotal columns: {len(data)}")
print(f"Total rows (before cleaning): {len(data['landslide_dist'])}")


Extracting target rasters...
  ✓ lsi_continuous: shape (10553, 12545) — OK
  ✓ lsi_classified: shape (10553, 12545) — OK

Total columns: 15
Total rows (before cleaning): 132387385


In [13]:
# ---- CREATE VALID PIXEL MASK ----
print("Creating valid pixel mask...")

# Start with all True
valid_mask = np.ones(len(data["pixel_row"]), dtype=bool)

# Mark False wherever ANY column has NaN
for name, arr in data.items():
    if name in ["pixel_row", "pixel_col"]:
        continue  # integer arrays, no NaN
    nan_mask = np.isnan(arr)
    valid_mask &= ~nan_mask
    
valid_count = valid_mask.sum()
total_count = len(valid_mask)
print(f"  Total pixels: {total_count:,}")
print(f"  Valid pixels: {valid_count:,}")
print(f"  Removed: {total_count - valid_count:,}")

# ---- FILTER ALL ARRAYS USING MASK ----
print("\nFiltering to valid pixels only...")
data_filtered = {}
for name, arr in data.items():
    data_filtered[name] = arr[valid_mask]

# ---- FREE MEMORY ----
del data
import gc
gc.collect()

# ---- CREATE DATAFRAME FROM FILTERED DATA ----
print("Creating DataFrame...")
df_clean = pd.DataFrame(data_filtered)
del data_filtered
gc.collect()

print(f"  Shape: {df_clean.shape}")
print(f"  Columns: {list(df_clean.columns)}")
print(f"  Memory: {df_clean.memory_usage(deep=True).sum() / (1024**3):.2f} GB")
print(f"  NaN check: {df_clean.isna().sum().sum()}")

Creating valid pixel mask...
  Total pixels: 132,387,385
  Valid pixels: 51,429,575
  Removed: 80,957,810

Filtering to valid pixels only...
Creating DataFrame...
  Shape: (51429575, 15)
  Columns: ['pixel_row', 'pixel_col', 'landslide_dist', 'slope', 'lulc', 'elevation', 'road_dist', 'builtup_dist', 'land_value', 'water_dist', 'lineament_dist', 'lst', 'railway_dist', 'lsi_continuous', 'lsi_classified']
  Memory: 2.87 GB
  NaN check: 0


In [15]:
# ---- FILTER LSI OUTLIERS ----
before = len(df_clean)
df_clean = df_clean[(df_clean["lsi_continuous"] >= 1.0) & (df_clean["lsi_continuous"] <= 5.0)].reset_index(drop=True)
print(f"LSI outlier filter: removed {before - len(df_clean):,} rows")

LSI outlier filter: removed 0 rows


In [17]:
# ---- CONVERT LULC TO INTEGER ----
df_clean["lulc"] = df_clean["lulc"].astype(np.int8)

In [18]:
# ---- CONVERT COORDINATES TO INTEGER ----
df_clean["pixel_row"] = df_clean["pixel_row"].astype(np.int32)
df_clean["pixel_col"] = df_clean["pixel_col"].astype(np.int32)

In [19]:
# ---- VERIFY ----
print(f"\nFinal shape: {df_clean.shape}")
print(f"Columns: {list(df_clean.columns)}")
print(f"LSI range: {df_clean['lsi_continuous'].min():.4f} to {df_clean['lsi_continuous'].max():.4f}")
print(f"LULC dtype: {df_clean['lulc'].dtype}")
print(f"Coordinate ranges: row [{df_clean['pixel_row'].min()}-{df_clean['pixel_row'].max()}], col [{df_clean['pixel_col'].min()}-{df_clean['pixel_col'].max()}]")


Final shape: (51423700, 15)
Columns: ['pixel_row', 'pixel_col', 'landslide_dist', 'slope', 'lulc', 'elevation', 'road_dist', 'builtup_dist', 'land_value', 'water_dist', 'lineament_dist', 'lst', 'railway_dist', 'lsi_continuous', 'lsi_classified']
LSI range: 2.0400 to 5.0000
LULC dtype: int8
Coordinate ranges: row [2-10551], col [1-12543]


In [20]:
# ---- SAVE AS PARQUET ----
parquet_path = r"D:\\Important\\Semester\\Semester X\\MTP\\LSI\\shimla_dataset.parquet"
df_clean.to_parquet(parquet_path, index=False, engine="pyarrow")
file_size = os.path.getsize(parquet_path) / (1024**3)
print(f"\nSaved to: {parquet_path}")
print(f"File size: {file_size:.2f} GB")


Saved to: D:\\Important\\Semester\\Semester X\\MTP\\LSI\\shimla_dataset.parquet
File size: 1.75 GB


In [21]:
# ---- FEATURE STATISTICS ----
print("=" * 60)
print("FEATURE STATISTICS (Input Variables)")
print("=" * 60)
feature_cols = ["landslide_dist", "slope", "lulc", "elevation", "road_dist",
                "builtup_dist", "land_value", "water_dist", "lineament_dist",
                "lst", "railway_dist"]
print(df_clean[feature_cols].describe().round(2).to_string())

FEATURE STATISTICS (Input Variables)
       landslide_dist        slope         lulc    elevation    road_dist  builtup_dist   land_value   water_dist  lineament_dist          lst  railway_dist
count     51423700.00  51423700.00  51423700.00  51423700.00  51423700.00   51423700.00  51423700.00  51423700.00     51423700.00  51423700.00   51423700.00
mean          3691.80        29.22         6.47      2382.89      1168.85        896.13       591.82      3939.90          767.63        24.39      49995.91
std           4237.21        11.09         4.22       837.26      2012.03       1207.06       198.82      3068.88          603.81         8.59      26098.45
min              0.00         0.00         1.00         0.00         0.00          0.00        14.78         0.00            0.00        -9.99          0.00
25%           1161.08        21.68         2.00      1801.43       121.66        113.14       497.51      1502.66          277.31        19.96      30629.35
50%           2432.98

In [22]:
# ---- TARGET STATISTICS ----
print("\n" + "=" * 60)
print("TARGET STATISTICS")
print("=" * 60)
print("\nContinuous LSI:")
print(df_clean["lsi_continuous"].describe().round(4).to_string())


TARGET STATISTICS

Continuous LSI:
count    5.142370e+07
mean     3.660600e+00
std      2.932000e-01
min      2.040000e+00
25%      3.480000e+00
50%      3.678000e+00
75%      3.864000e+00
max      5.000000e+00


In [23]:
print("\nClassified LSI — Class Distribution:")
class_counts = df_clean["lsi_classified"].value_counts().sort_index()
class_labels = {1: "Very Low", 2: "Low", 3: "Moderate", 4: "High", 5: "Very High"}
print(f"  {'Class':<8} {'Label':<12} {'Count':>12} {'Percentage':>10}")
print("  " + "-" * 45)
for cls, count in class_counts.items():
    pct = (count / len(df_clean)) * 100
    label = class_labels.get(int(cls), "Unknown")
    print(f"  {int(cls):<8} {label:<12} {count:>12,} {pct:>9.2f}%")
print(f"\n  Total: {len(df_clean):,}")


Classified LSI — Class Distribution:
  Class    Label               Count Percentage
  ---------------------------------------------
  1        Very Low        1,060,985      2.06%
  2        Low            13,055,064     25.39%
  3        Moderate       31,455,011     61.17%
  4        High            5,826,106     11.33%
  5        Very High          26,534      0.05%

  Total: 51,423,700


In [ ]:
# # ---- SAVE DATASET ----
# output_path = r"D:\\Important\\Semester\\Semester X\\MTP\\LSI\\shimla_ml_dataset.csv"

# print("Saving dataset...")
# df_clean.to_csv(output_path, index=False)

# file_size = os.path.getsize(output_path) / (1024**3)
# print(f"  Saved to: {output_path}")
# print(f"  File size: {file_size:.2f} GB")
# print(f"  Rows: {len(df_clean):,}")
# print(f"  Columns: {len(df_clean.columns)}")
# print(f"  Column names: {list(df_clean.columns)}")

Saving dataset...
  Saved to: D:\\Important\\Semester\\Semester X\\MTP\\LSI\\shimla_ml_dataset.csv
  File size: 5.27 GB
  Rows: 51,423,700
  Columns: 13
  Column names: ['landslide_dist', 'slope', 'lulc', 'elevation', 'road_dist', 'builtup_dist', 'land_value', 'water_dist', 'lineament_dist', 'lst', 'railway_dist', 'lsi_continuous', 'lsi_classified']


In [52]:
# output_path = r"D:\\Important\\Semester\\Semester X\\MTP\\LSI\\shimla_ml_dataset.csv"

In [ ]:
# # ---- SAVE AS PARQUET ----
# parquet_path = r"D:\\Important\\Semester\\Semester X\\MTP\\LSI\\shimla_ml_dataset.parquet"

# print("Saving as Parquet...")
# df_clean.to_parquet(parquet_path, index=False, engine="pyarrow")

# file_size = os.path.getsize(parquet_path) / (1024**3)
# csv_size = os.path.getsize(output_path) / (1024**3)

# print(f"  Saved to: {parquet_path}")
# print(f"  Parquet size: {file_size:.2f} GB")
# print(f"  CSV size: {csv_size:.2f} GB")
# print(f"  Compression ratio: {csv_size/file_size:.1f}x smaller")

Saving as Parquet...
  Saved to: D:\\Important\\Semester\\Semester X\\MTP\\LSI\\shimla_ml_dataset.parquet
  Parquet size: 1.67 GB
  CSV size: 5.27 GB
  Compression ratio: 3.2x smaller


In [24]:
import rasterio

# ---- CHECK PIXEL SIZE FROM RASTER ----
raster_path = feature_files["landslide_dist"]

with rasterio.open(raster_path) as src:
    pixel_width = src.res[0]   # x resolution
    pixel_height = src.res[1]  # y resolution
    crs = src.crs
    
print(f"CRS: {crs}")
print(f"CRS units: meters (UTM)")
print(f"Pixel width: {pixel_width} m")
print(f"Pixel height: {pixel_height} m")
print(f"Pixel area: {pixel_width * pixel_height} sq.m")

# ---- STUDY AREA DIMENSIONS ----
row_range = df_clean["pixel_row"].max() - df_clean["pixel_row"].min()
col_range = df_clean["pixel_col"].max() - df_clean["pixel_col"].min()

print(f"\nStudy area extent:")
print(f"  Rows: {row_range} pixels = {row_range * pixel_height / 1000:.2f} km")
print(f"  Cols: {col_range} pixels = {col_range * pixel_width / 1000:.2f} km")
print(f"  Bounding box area: {(row_range * pixel_height / 1000) * (col_range * pixel_width / 1000):.2f} sq.km")

# ---- BLOCK SIZE CALCULATION ----
block_size_m = 1000  # 1 km
block_size_px = int(block_size_m / pixel_width)

print(f"\nProposed block size:")
print(f"  {block_size_m} m = {block_size_px} pixels")
print(f"  Estimated blocks across rows: {row_range // block_size_px}")
print(f"  Estimated blocks across cols: {col_range // block_size_px}")
print(f"  Estimated total blocks: {(row_range // block_size_px) * (col_range // block_size_px)}")

CRS: EPSG:32643
CRS units: meters (UTM)
Pixel width: 10.0 m
Pixel height: 10.0 m
Pixel area: 100.0 sq.m

Study area extent:
  Rows: 10549 pixels = 105.49 km
  Cols: 12542 pixels = 125.42 km
  Bounding box area: 13230.56 sq.km

Proposed block size:
  1000 m = 100 pixels
  Estimated blocks across rows: 105
  Estimated blocks across cols: 125
  Estimated total blocks: 13125
